In [33]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from itertools import combinations
from collections import Counter

In [34]:
s = pd.read_csv('sales_transactions_cleaned.csv')
c = pd.read_csv('customers_cleaned.csv')
s['revenue'] =  (s['quantity']*s['price'])- pd.to_numeric(s['discount_amount'],errors='coerce').fillna(0)

In [35]:
feat = s.groupby('customer_id').agg(total_purchases = ('transaction_id', 'nunique') , avg_purchase_value = ('revenue' , 'mean')).reset_index()

display(feat.head())

,customer_id,total_purchases,avg_purchase_value
0,101,13,8.484615
1,102,17,19.315294
2,103,22,24.063182
3,104,21,8.260000
4,105,11,13.642727


In [36]:
scaler = StandardScaler() 
X = scaler.fit_transform(feat[['total_purchases', 'avg_purchase_value']])

kmean = KMeans(n_clusters=3, random_state=42, n_init=10)
feat['cluster_label'] = kmean.fit_predict(X) + 1

print(feat['cluster_label'].value_counts().sort_index())

cluster_label
1    255
2    286
3     79
Name: count, dtype: int64


In [37]:
pairpro = s.groupby('transaction_id')['product_id'].apply(list)

paircount = Counter()
for products in pairpro:
    if len(products) >= 2:
        for pair in combinations(sorted(products), 2):
            paircount[pair] += 1
allproduct = s['product_id'].dropna().unique()
affinity = {}
for pid in allproduct:
    related = {}
    for (a,b), cnt in paircount.items():
        if a == pid: related[b] = cnt
        if b == pid: related[a] = cnt
    affinity[pid] = sorted(related, key=related.get, reverse=True)[:3]

print(affinity.get(1,[]))

[]


In [38]:
custcluster = feat[['customer_id','cluster_label']]

purchased = s.groupby('customer_id')['product_id'].apply(set).to_dict()

s_cluster = s.merge(custcluster, on='customer_id')
clustertop = (s_cluster.groupby(['cluster_label', 'product_id'])['quantity'].sum().reset_index().sort_values(['cluster_label','quantity'], ascending=[True,False]))

rows = []
for _, row in custcluster.iterrows():
    cid, cl = row['customer_id'], row['cluster_label']
    bought = purchased.get(cid,set())
    recs = [int(p) for p in clustertop[clustertop['cluster_label']==cl]['product_id']
           if p not in bought][:3]
    while len(recs) < 3: recs.append(None)
    rows.append([int(cid), cl, recs[0], recs[1], recs[2]])

result = pd.DataFrame(rows, columns=['customers_id','cluster_label','recommended_product_1','recommended_product_2','recommended_product_3'])
display(result.head())
result.to_csv('Session5_Segmentation_and_RecommendationsVTest.csv', index=False)
print('Success')

,customers_id,cluster_label,recommended_product_1,recommended_product_2,recommended_product_3
0,101,1,6.0,3.0,2.0
1,102,2,9.0,18.0,20.0
2,103,2,9.0,18.0,20.0
3,104,2,9.0,18.0,20.0
4,105,1,3.0,2.0,9.0


Success
